# Whisper large-v3 — ASR runner

Runs on a GPU session, separately from the main notebook, because Step 3's
Whisper sweep is the only GPU work left and it should not wait behind anything.

Two passes, and the difference between them is a result:

* **per-segment** — cut at the diarized turn boundaries and transcribe each.
  Held identical to how Sarvam is run, so the two systems are compared on the
  recogniser alone.
* **long-form** — one pass with word timestamps, words attributed by time
  overlap. Whisper can do this and Saaras cannot, so running both on Whisper
  isolates the strategy effect rather than confounding it with the system.

Turns come from **reverb-v2**: 2,650 merged segments at a median 7.66 s, against
community-1's 10,279 at 1.13 s. Fewer, longer segments mean the recogniser gets
context rather than fragments.

No language is ever passed in — `language=None` lets Whisper detect it, because
the reference script is ground truth and hinting from it would be a leak.

**Settings:** Internet ON, GPU T4, and attach both the audio dataset and the
main notebook's saved output (for the Step 2 RTTMs).


In [ ]:
# faster-whisper only. NOT whisperx: it pins an older pyannote, and the only
# part of it we want -- assigning words to speakers -- is asr.assign_words().
# faster-whisper is CTranslate2 underneath and pulls no pyannote at all, so it
# coexists with whatever this session already has.
%pip install -q faster-whisper

import subprocess, sys, shutil
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/ParvGoyal08/MultilingualASR.git"
CODE_DIR = Path("/kaggle/working/sarvam-assignment" if IN_KAGGLE else "./sarvam-assignment")

def sync():
    if (CODE_DIR / ".git").exists():
        if subprocess.run(["git","-C",str(CODE_DIR),"fetch","-q","origin","main"]).returncode == 0:
            subprocess.run(["git","-C",str(CODE_DIR),"reset","-q","--hard","origin/main"], check=True)
            return
        shutil.rmtree(CODE_DIR, ignore_errors=True)
    subprocess.run(["git","clone","-q",REPO_URL,str(CODE_DIR)], check=True)

sync()
print("code @", subprocess.run(["git","-C",str(CODE_DIR),"log","-1","--format=%h  %s"],
                               capture_output=True, text=True).stdout.strip())
for m in [k for k in list(sys.modules) if k == "sarvam_diar" or k.startswith("sarvam_diar.")]:
    del sys.modules[m]
sys.path.insert(0, str(CODE_DIR))

import torch
from sarvam_diar.config import Config, StageFlags
from sarvam_diar import data, asr, diarization, utils
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert hasattr(asr, "run_segmented"), "stale clone -- restart the kernel and re-run"


## Paths and inputs

In [ ]:
ROOT = Path("/kaggle/working/sarvam_diarization") if IN_KAGGLE else Path("local_out")
ROOT.mkdir(parents=True, exist_ok=True)

if IN_KAGGLE:
    # Same relink as the main notebook: /kaggle/input moves when inputs change.
    print("dataset:", utils.relink_dataset(ROOT, "/kaggle/input"))
    # The Step 2 RTTMs must be here too -- attach the main notebook's saved
    # output, since Whisper is transcribed over ITS turns.
    prev = next((p.parent for p in sorted(Path("/kaggle/input").rglob("hypotheses"))), None)
    if prev:
        import shutil as _sh
        for sub in ("hypotheses", "reference", "results"):
            src = prev / sub
            if src.exists():
                _sh.copytree(src, ROOT / sub, dirs_exist_ok=True)
        print("restored Step 2 checkpoints from", prev)

cfg = Config.create(root=ROOT, work_dir=Path("/kaggle/working/tmp") if IN_KAGGLE else Path("local_out/.work"))
clips = data.parse_ground_truth(data.load_segments_csv(cfg))
inputs, _ = data.split_reference(clips, None, cfg=cfg)

import dataclasses
ready = [dataclasses.replace(ci, wav_path=str(cfg.wav_path(cid)))
         for cid, ci in inputs.items() if cfg.wav_path(cid).exists()]
DIAR = "reverb-v2"
have = sum(1 for c in ready if diarization.is_done(cfg, DIAR, c.clip_id))
print(f"{len(ready)} clips with audio; {have} have {DIAR} turns")
assert have, f"no {DIAR} hypotheses found -- attach the main notebook's output"


## Smoke gate — measure RTF before the sweep

In [ ]:
# --- SMOKE GATE: measure before committing to the full sweep -----------------
# Same discipline as the DiariZen runner. Per-segment Whisper makes one model
# call per turn, and there are ~2,650 of them, so per-call overhead matters more
# than audio duration -- which is exactly the thing an RTF guess gets wrong.
import time, soundfile as sf

probe = sorted(ready, key=lambda c: c.duration)
probe = [probe[0], probe[len(probe)//2], probe[-1]]
print(f"{'clip':<26}{'dur':>8}{'segs':>6}{'elapsed':>9}{'rtf':>7}{'words':>7}  lang")
rows = []
for c in probe:
    turns = asr.merge_same_speaker(diarization.load_hypothesis(cfg, DIAR, c.clip_id), 1.0)
    t = time.time()
    segs, meta = asr.transcribe_segments(cfg, "whisper-large-v3",
                                         Path(c.wav_path), turns)
    el = time.time() - t
    nw = sum(len(s["text"].split()) for s in segs)
    rows.append((c.duration, el, el / c.duration, len(segs), nw))
    print(f"{c.clip_id:<26}{c.duration:8.0f}{len(segs):6d}{el:9.1f}{el/c.duration:7.3f}"
          f"{nw:7d}  {meta.get('detected_language')}")

problems = []
if all(r[4] == 0 for r in rows):
    problems.append("every probe clip returned zero words")
if any(r[3] == 0 for r in rows):
    problems.append("a clip produced no segments -- are the reverb-v2 turns present?")
if problems:
    for x in problems:
        print("  -", x)
    raise SystemExit("SMOKE GATE FAILED -- do not start the sweep")

mean_rtf = sum(r[2] for r in rows) / len(rows)
total = sum(c.duration for c in ready)
print(f"\nmean RTF {mean_rtf:.3f} (spread {min(r[2] for r in rows):.3f}-{max(r[2] for r in rows):.3f})")
print(f"projected per-segment sweep: {mean_rtf*total/3600:.1f} h for {len(ready)} clips")
print("Kaggle GPU sessions cap at 12 h. Both passes below are resumable, so a")
print("session that ends early costs only the clip in flight.")


## Pass 1 — per-segment (comparable to Sarvam)

In [ ]:
# Per-segment, over the SAME turns Sarvam used, so the two systems differ only
# in the recogniser. Whisper could do long-form, and does in the next cell --
# but the comparison against Sarvam has to hold the strategy fixed, because
# Saaras returns no word timings and cannot do long-form at all.
metrics_seg = asr.run_segmented(
    cfg, ready, diar_model=DIAR, systems=["whisper-large-v3"],
    flags=StageFlags(), merge_gap=1.0,
)
print(metrics_seg[["clip_id","n_segments","n_words","elapsed_sec","rtf",
                   "detected_language"]].head(10).to_string(index=False))


## Pass 2 — long-form (isolates the strategy effect)

In [ ]:
# Long-form: one pass over the whole clip with word timestamps, then words are
# attributed by time overlap. Only Whisper can do this, so running it on the
# same clips isolates the STRATEGY effect on one system instead of mixing it
# into the system comparison.
metrics_lf = asr.run(cfg, ready, flags=StageFlags(), systems=["whisper-large-v3"])
print(metrics_lf[["clip_id","n_words","elapsed_sec","rtf","detected_language"]].head(10).to_string(index=False))


## Keep the output

In [ ]:
import shutil, os
if IN_KAGGLE:
    z = shutil.make_archive("/kaggle/working/asr_whisper", "zip", root_dir=str(ROOT / "asr"))
    print(z, f"({os.path.getsize(z)/1e6:.1f} MB)")
    print("Save Version, then attach this output to the main notebook to score it.")
